# Implement KV-Cache from Scratch
### Problem Statement
During autoregressive text generation, the model generates one token at a time. Without caching, at step `t` we'd recompute attention over all `t` previous tokens — `O(t²)` total work for generating a sequence of length `T`.

The **KV-Cache** eliminates this redundancy: we cache the K and V projections of all previous tokens. At each new step, we only compute Q/K/V for the **new token** and append K/V to the cache.

This reduces per-step attention from `O(t × d)` to `O(d)` for the projection, making generation **much** faster.

Your goal is to implement an attention module with KV-Cache and show the speedup.

---

### Requirements

1. **CachedAttention Module**
   - On first call (prefill): compute full Q, K, V and initialize cache.
   - On subsequent calls (decode): compute Q, K, V for new token only, append K/V to cache.
   - Attention is always computed over the full cached K/V sequence.

2. **Cache Management**
   - Store K and V caches as tensors of shape `(batch, num_heads, cached_len, d_head)`.
   - Grow the cache by concatenating new K/V along the sequence dimension.

3. **Validate**
   - The cached incremental output at each step must match the non-cached full recomputation.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Cache must grow correctly across multiple decode steps.
- ✅ Incremental output must match full recomputation.

---

<details>
  <summary>💡 Hint</summary>

  - At decode time, `q` has shape `(batch, 1, d_model)` (single new token).
  - Use `torch.cat([cache_k, new_k], dim=2)` to grow the cache along the seq dimension.
  - The output at decode time is `(batch, 1, d_model)` — attention over all cached tokens for the new query.
  - For validation, run the full sequence through non-cached attention and compare the last token's output.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class CachedMultiHeadAttention(nn.Module):
    """
    Multi-Head Attention with KV-Cache for efficient autoregressive generation.
    """
    def __init__(self, num_heads: int, d_model: int):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads

        self.Q_w = nn.Linear(d_model, d_model, bias=False)
        self.K_w = nn.Linear(d_model, d_model, bias=False)
        self.V_w = nn.Linear(d_model, d_model, bias=False)
        self.W_out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cache_k=None, cache_v=None):
        """
        Args:
            x (Tensor): Input of shape (batch, seq_len, d_model)
                        seq_len=full for prefill, seq_len=1 for decode
            cache_k (Tensor, optional): Cached keys (batch, num_heads, cached_len, d_head)
            cache_v (Tensor, optional): Cached values (batch, num_heads, cached_len, d_head)

        Returns:
            output (Tensor): (batch, seq_len, d_model)
            new_cache_k (Tensor): Updated key cache
            new_cache_v (Tensor): Updated value cache
        """
        # TODO: Implement forward with KV-Cache
        # 1. Project Q, K, V from x
        # 2. Reshape to (batch, num_heads, seq_len, d_head)
        # 3. If cache exists, concatenate new K/V with cache
        # 4. Compute attention: Q attends to full K/V (cached + new)
        # 5. Return output and updated caches
        ...

In [ ]:
# Validate: incremental decode must match full recomputation
torch.manual_seed(42)
batch_size, seq_len, d_model, num_heads = 2, 6, 8, 2
tokens = torch.randn(batch_size, seq_len, d_model)

attn = CachedMultiHeadAttention(num_heads, d_model)
attn.eval()

# Method 1: Full forward (no cache) — the reference
with torch.no_grad():
    full_output, _, _ = attn(tokens)

# Method 2: Incremental with KV-Cache — token by token
cache_k, cache_v = None, None
incremental_outputs = []
with torch.no_grad():
    for t in range(seq_len):
        token_t = tokens[:, t:t+1, :]  # (batch, 1, d_model)
        out_t, cache_k, cache_v = attn(token_t, cache_k, cache_v)
        incremental_outputs.append(out_t)

# The last incremental output should match the last position of full output
last_full = full_output[:, -1:, :]
last_incremental = incremental_outputs[-1]

print(f"Full output (last token):        {last_full[0, 0, :4]}")
print(f"Incremental output (last token): {last_incremental[0, 0, :4]}")

assert torch.allclose(last_full, last_incremental, atol=1e-6)
print(f"\nCache shape: K={cache_k.shape}, V={cache_v.shape}")
print("\n✅ KV-Cache implementation is correct!")